In [14]:
import os
from dotenv import load_dotenv
load_dotenv(".env", override=True)

LLM_MODEL = os.getenv("LLM_MODEL", "gemini/gemini-flash-latest") 
# A container tag namespaces a user's memories inside Supermemory (multi-tenant isolation).
USER_ID = "loan_applicants"

print("Google key      :", bool(os.getenv("GEMINI_API_KEY")), "(required for the LLM)")
print("Supermemory key :", bool(os.getenv("SUPERMEMORY_API_KEY")), "(required for the memory store)")
print("LLM model       :", LLM_MODEL)
print("Memory namespace:", USER_ID)

Google key      : True (required for the LLM)
Supermemory key : True (required for the memory store)
LLM model       : gemini/gemini-flash-latest
Memory namespace: loan_applicants


In [15]:
# Readiness check: confirms libraries import and keys are present. Calls no paid API.
status = {}
for name, module in [("litellm", "litellm"), ("supermemory", "supermemory"), ("tiktoken", "tiktoken")]:
    try:
        __import__(module); status[name] = "ok"
    except Exception as e:
        status[name] = f"import failed: {e}"
status["GEMINI_API_KEY"]      = "ok" if os.getenv("GEMINI_API_KEY") else "missing (LLM will fail)"
status["SUPERMEMORY_API_KEY"] = "ok" if os.getenv("SUPERMEMORY_API_KEY") else "missing (memory store will fail)"

print("Environment readiness")
print("---------------------")
for k, v in status.items():
    print(f"  {k:>20} : {v}")

Environment readiness
---------------------
               litellm : ok
           supermemory : ok
              tiktoken : ok
        GEMINI_API_KEY : ok
   SUPERMEMORY_API_KEY : ok


## Write the long-term memories

In [ ]:
import supermemory
mem = supermemory.Supermemory(api_key=os.getenv("SUPERMEMORY_API_KEY"))

# Write a memory into the store (each is stored as a document, tagged with its applicant ID).
def remember(text, applicant_id, **extra):
    return mem.add(content=text, container_tag=USER_ID, metadata={"Applicant ID": applicant_id, **extra})

def recall(query, app_id, k=1):
    client = mem
    out = []
    for r in client.search.memories(q=query, container_tag=USER_ID, filters={"key": "Applicant ID", "value": app_id }, limit=k).results:
        if getattr(r, "memory", None):
            out.append(((r.metadata or {}).get("Applicant ID", "?"), r.memory, float(r.similarity or 0.0)))
    if not out:
        for r in client.search.documents(q=query, container_tags=[USER_ID], filters={"key": "Applicant ID", "value": app_id}, limit=k).results:
            text = " ".join(c.content for c in (r.chunks or []) if c and c.content).strip()
            if text:
                out.append(((r.metadata or {}).get("Applicant ID", "?"), text, float(r.score or 0.0)))
    return out if out else (None, None, 0.0)

In [29]:
# load a json file which has a list of dictionaries, each representing a past decision
import json

with open("./data/wealthpilot/mock_api/past_decisions.json", "r") as f:
    past_decisions_json = json.load(f)

In [ ]:
for decision in past_decisions_json[0:10]:
    remember(str(decision), decision['applicant_id'])

In [38]:
for applicant_id, text, score in recall("Are there any past history of applicant ASH-A-00037?", k=1):
    print(text)

{'decision_id': 'ASH-D-00000', 'applicant_id': 'ASH-A-00037', 'decided_on': '2025-07-05', 'outcome': 'declined', 'internal_grade': 'A1', 'approved_amount_inr': 1500000, 'reason_code': 'ADVERSE_BUREAU', 'human_signatory': 'ASH-U-002'}
